In [0]:
from pyspark.sql import functions as F

In [0]:
%sql

DROP DATABASE IF EXISTS workspace.silver CASCADE;

In [0]:
%sql

CREATE DATABASE IF NOT EXISTS workspace.silver
COMMENT 'Capa Silver: terremotos USGS limpios y normalizados';

In [0]:
df = spark.table(
    "workspace.bronze.usgs_earthquakes"
)

In [0]:
df_silver = (
    df

    # Tiempo
    .withColumn(
        "event_timestamp",
        F.to_timestamp(
            F.from_unixtime(
                F.col("event_time") / 1000
            )
        )
    )

    .withColumn(
        "updated_timestamp",
        F.to_timestamp(
            F.from_unixtime(
                F.col("updated_time") / 1000
            )
        )
    )

    # Flag tsunami
    .withColumn(
        "tsunami_flag",
        F.when(
            F.col("tsunami") == 1,
            True
        ).otherwise(False)
    )

    # Limpieza de texto
    .withColumn(
        "place",
        F.trim(
            F.col("place")
        )
    )

    .withColumn(
        "alert",
        F.coalesce(
            F.col("alert"),
            F.lit("No alert")
        )
    )

    .withColumn(
        "status",
        F.coalesce(
            F.col("status"),
            F.lit("unknown")
        )
    )

    .withColumn(
        "network",
        F.coalesce(
            F.col("network"),
            F.lit("UNKNOWN")
        )
    )

    .withColumn(
        "magnitude_type",
        F.coalesce(
            F.col("magnitude_type"),
            F.lit("unknown")
        )
    )

    .withColumn(
        "event_type",
        F.coalesce(
            F.col("event_type"),
            F.lit("unknown")
        )
    )

    # Eliminar duplicados
    .dropDuplicates(
        ["earthquake_id"]
    )
)

In [0]:
df_silver = df_silver.select(
    "earthquake_id",
    "magnitude",
    "place",
    "event_timestamp",
    "updated_timestamp",
    "felt",
    "cdi",
    "mmi",
    "alert",
    "status",
    "tsunami_flag",
    "significance",
    "network",
    "event_code",
    "ids",
    "sources",
    "product_types",
    "station_count",
    "distance_min",
    "rms",
    "azimuthal_gap",
    "magnitude_type",
    "event_type",
    "longitude",
    "latitude",
    "depth"
)

In [0]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "mergeSchema",
        "true"
    ) \
    .saveAsTable(
        "workspace.silver.usgs_earthquakes"
    )